## Feature concatanation

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "preprocess":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_INCIDENTS_PATH = PROJECT_ROOT / "data" / "raw" / "incidents.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_INCIDENTS_PATH = PROCESSED_DATA_DIR / "incidents_feature_engineered.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

incidents_df = pd.read_csv(RAW_INCIDENTS_PATH)

incidents_df["feature_concatanation"] = (
    incidents_df["reporter_role"].fillna("").str.strip()
    + " "
    + incidents_df["incident_description"].fillna("").str.strip()
)
incidents_df["feature_concatanation"] = incidents_df["feature_concatanation"].str.strip()

incidents_df = incidents_df[[
    "date",
    "time",
    "reporter_role",
    "assigned_department",
    "incident_description",
    "urgency",
    "feature_concatanation",
]]

incidents_df.to_csv(PROCESSED_INCIDENTS_PATH, index=False)

print(f"Loaded {len(incidents_df)} incidents from {RAW_INCIDENTS_PATH}")
print(f"Saved feature-engineered data to {PROCESSED_INCIDENTS_PATH}")
incidents_df[[
    "date",
    "time",
    "reporter_role",
    "assigned_department",
    "incident_description",
    "urgency",
    "feature_concatanation",
]].head()


Loaded 3600 incidents from /home/lakshan/ssp/IMS/data/raw/incidents.csv
Saved feature-engineered data to /home/lakshan/ssp/IMS/data/processed/incidents_feature_engineered.csv


,date,time,reporter_role,assigned_department,incident_description,urgency,feature_concatanation
0,2025-01-01,02:15 PM,Hospital Administrator,Finance and Account,Consultant session revenue-share needs recalib...,Low,Hospital Administrator Consultant session reve...
1,2025-01-01,07:10 AM,Staff Nurse - Paediatric Ward,Dietary and Food Services,Diet slip notes G6PD + seafood allergy for bed...,High,Staff Nurse - Paediatric Ward Diet slip notes ...
2,2025-01-01,07:10 AM,OPD Nursing Officer,Security,Large crowd surged at OPD entrance when tokens...,Medium,OPD Nursing Officer Large crowd surged at OPD ...
3,2025-01-01,07:35 AM,ICU Charge Nurse,Facility Management,ICU AHU-2 not holding 22C; temp fluctuating 24...,High,ICU Charge Nurse ICU AHU-2 not holding 22C; te...
4,2025-01-01,07:40 AM,"Nursing Officer-in-Charge, Surgical Ward",Human Resources,Public holiday OT for our ward isn’t appearing...,High,"Nursing Officer-in-Charge, Surgical Ward Publi..."


## Lable Encoding

In [2]:
DEPARTMENT_LABELS_PATH = PROCESSED_DATA_DIR / "department_label_mapping.csv"
ENCODED_INCIDENTS_PATH = PROCESSED_DATA_DIR / "incidents_feature_engineered_encoded.csv"

department_labels = pd.DataFrame(
    {
        "assigned_department": sorted(incidents_df["assigned_department"].dropna().unique())
    }
)
department_labels["department_label"] = range(len(department_labels))

department_to_label = dict(
    zip(department_labels["assigned_department"], department_labels["department_label"])
)

encoded_incidents_df = incidents_df.copy()
encoded_incidents_df["department_label"] = encoded_incidents_df["assigned_department"].map(department_to_label)

department_labels.to_csv(DEPARTMENT_LABELS_PATH, index=False)
encoded_incidents_df.to_csv(ENCODED_INCIDENTS_PATH, index=False)

print(f"Saved department label mapping to {DEPARTMENT_LABELS_PATH}")
print(f"Saved encoded incidents to {ENCODED_INCIDENTS_PATH}")
encoded_incidents_df[[
    "assigned_department",
    "department_label",
    "feature_concatanation",
]].head()


Saved department label mapping to /home/lakshan/ssp/IMS/data/processed/department_label_mapping.csv
Saved encoded incidents to /home/lakshan/ssp/IMS/data/processed/incidents_feature_engineered_encoded.csv


,assigned_department,department_label,feature_concatanation
0,Finance and Account,3,Hospital Administrator Consultant session reve...
1,Dietary and Food Services,1,Staff Nurse - Paediatric Ward Diet slip notes ...
2,Security,9,OPD Nursing Officer Large crowd surged at OPD ...
3,Facility Management,2,ICU Charge Nurse ICU AHU-2 not holding 22C; te...
4,Human Resources,5,"Nursing Officer-in-Charge, Surgical Ward Publi..."


## Shuffle and Split Dataset

In [3]:
TRAIN_SPLIT_PATH = PROCESSED_DATA_DIR / "incidents_train.csv"
VALIDATION_SPLIT_PATH = PROCESSED_DATA_DIR / "incidents_validation.csv"
TEST_SPLIT_PATH = PROCESSED_DATA_DIR / "incidents_test.csv"

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

if round(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO, 10) != 1.0:
    raise ValueError("Train, validation, and test ratios must sum to 1.0")

shuffled_incidents_df = encoded_incidents_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_end_index = int(len(shuffled_incidents_df) * TRAIN_RATIO)
validation_end_index = train_end_index + int(len(shuffled_incidents_df) * VALIDATION_RATIO)

train_df = shuffled_incidents_df.iloc[:train_end_index].copy()
validation_df = shuffled_incidents_df.iloc[train_end_index:validation_end_index].copy()
test_df = shuffled_incidents_df.iloc[validation_end_index:].copy()

train_df.to_csv(TRAIN_SPLIT_PATH, index=False)
validation_df.to_csv(VALIDATION_SPLIT_PATH, index=False)
test_df.to_csv(TEST_SPLIT_PATH, index=False)

print(f"Saved shuffled train split to {TRAIN_SPLIT_PATH} with {len(train_df)} rows")
print(f"Saved shuffled validation split to {VALIDATION_SPLIT_PATH} with {len(validation_df)} rows")
print(f"Saved shuffled test split to {TEST_SPLIT_PATH} with {len(test_df)} rows")

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "ratio": [
            len(train_df) / len(shuffled_incidents_df),
            len(validation_df) / len(shuffled_incidents_df),
            len(test_df) / len(shuffled_incidents_df),
        ],
    }
)


Saved shuffled train split to /home/lakshan/ssp/IMS/data/processed/incidents_train.csv with 2520 rows
Saved shuffled validation split to /home/lakshan/ssp/IMS/data/processed/incidents_validation.csv with 540 rows
Saved shuffled test split to /home/lakshan/ssp/IMS/data/processed/incidents_test.csv with 540 rows


,split,rows,ratio
0,train,2520,0.70
1,validation,540,0.15
2,test,540,0.15
